In [1]:
import logging
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"                   # sin mensajes informativos de MLflow
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"   # sin barras de progreso al cargar el modelo
# Permite importar el paquete `proyecto` sin instalarlo: el código vive en src/
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

# 09 · Inferencia — del modelo registrado a una función que cualquiera puede usar

El modelo vive en el registro de MLflow (notebook 08); la app **no consulta MLflow ni Databricks**. Para que alguien que no
sabe nada de esto pueda usarlo hacen falta cuatro cosas (las del ejemplo del curso):

1. **Exportar** el modelo `champion` a un archivo (`models/model.joblib`) que viaja con el repositorio.
2. **Un esquema de entrada** (`models/input_schema.json`): qué datos se piden, qué valores son *posibles*, perfiles de
   ejemplo y la regla de decisión.
3. **Una función `predict` que valida**: nunca responde en silencio con un dato imposible.
4. **La ficha del modelo** (`models/ficha_modelo.md`): qué es, cuánto rinde (con intervalo), dónde falla y para qué **no** usarlo.

Y una revisión final: que el código esté **completamente modularizado** y la app **corra con el modelo real**, lista para el
despliegue en Streamlit.

### `predict.py`: para qué sirve y qué se le agregó

Es el **puente entre el modelo y quien lo usa**: la app y este notebook no llaman al modelo directamente, llaman a
`predict_with_scores`. Valida las entradas (columnas, tipos, rangos posibles), predice y aplica la regla de decisión. La
plantilla del curso lo trae genérico (decide por la clase más probable o por una matriz de costos fija). Para este caso se
le agregaron dos cosas, con pruebas en `tests/test_predict_retiro.py`:

| Agregado | Por qué |
|---|---|
| `prepare_inputs` | El modelo usa 3 indicadores de tenencia de producto que se deducen de los saldos: quien usa la app no los digita |
| Regla de **valor esperado** (`apply_expected_value_rule`) | La decisión del negocio depende del saldo de cada asociado: contactar si *probabilidad × 20 % × saldo de aportes > $30.000* |

## 0. Conexión, modelo registrado y resultados del 07

In [2]:
import json
import subprocess
from datetime import datetime

import joblib
import mlflow
import numpy as np
import pandas as pd

from proyecto.config import (BINARY_INPUTS, CONTACT_COST, INPUT_LABELS, POSSIBLE_RANGE, PROCESSED_DIR, PRODUCT_FLAGS,
                             REPORTS_DIR, RETENTION_RATE, ROOT, TARGET)
from proyecto.inference.predict import InputValidationError, load_model, predict_with_scores
from proyecto.inference.schema import build_schema, load_schema, save_schema
from proyecto.tracking.mlflow_utils import configure_mlflow, registered_model_name
from proyecto.viz import fmt_num, fmt_pct, table

logging.getLogger("mlflow").setLevel(logging.ERROR)
pd.set_option("display.width", 170)
MODELS_DIR = ROOT / "models"
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    where = configure_mlflow()
registry = json.loads((REPORTS_DIR / "08_registro.json").read_text(encoding="utf-8"))
results = json.loads((REPORTS_DIR / "07_resultados.json").read_text(encoding="utf-8"))
decisions = json.loads((REPORTS_DIR / "07_decisiones_congeladas.json").read_text(encoding="utf-8"))
MODEL_NAME = registered_model_name("retiro_asociados_rf")
MODEL_URI = f"models:/{MODEL_NAME}@champion"
INPUT_COLUMNS = mlflow.models.get_model_info(MODEL_URI).signature.inputs.input_names()   # la firma registrada (08)
RAW_INPUTS = [c for c in INPUT_COLUMNS if c not in PRODUCT_FLAGS]       # las que digita una persona
assert INPUT_COLUMNS == registry["columnas_de_entrada"]          # las mismas que anotó el notebook 08
train = joblib.load(PROCESSED_DIR / "train.joblib")
print(f"MLflow: {where} · modelo: {MODEL_NAME} versión {registry['version']} (alias {registry['alias']})")
print(f"Entradas de la firma registrada: {len(INPUT_COLUMNS)} · las que pide la app: {len(RAW_INPUTS)} "
      f"(los {len(INPUT_COLUMNS) - len(RAW_INPUTS)} indicadores de tenencia se calculan con predict.prepare_inputs)")

MLflow: local · modelo: retiro_asociados_rf versión 1 (alias champion)
Entradas de la firma registrada: 22 · las que pide la app: 19 (los 3 indicadores de tenencia se calculan con predict.prepare_inputs)


## 1. Exportar el modelo `champion`

Se pide al registro el modelo **por su alias** y se guarda en un archivo. Se exporta desde `champion` (no desde el
`final_pipeline.joblib` del notebook 07) para tener la certeza de que se entrega **lo que quedó registrado**. Se guarda
comprimido (sin pérdida): GitHub rechaza archivos de más de 100 MB.

In [3]:
model = mlflow.sklearn.load_model(MODEL_URI)
joblib.dump(model, MODELS_DIR / "model.joblib", compress=3)
size_mb = (MODELS_DIR / "model.joblib").stat().st_size / 1e6
print(f"models/model.joblib guardado: {fmt_num(size_mb, 1)} MB (pipeline sin comprimir: "
      f"{fmt_num((MODELS_DIR / 'final_pipeline.joblib').stat().st_size / 1e6, 1)} MB; límite de GitHub: 100 MB)")
assert size_mb < 100

models/model.joblib guardado: 17,6 MB (pipeline sin comprimir: 54,6 MB; límite de GitHub: 100 MB)


## 2. El esquema de entrada

La app y `predict` necesitan saber qué se puede preguntar. Dos rangos distintos, como pide la guía:

- **Rango observado:** lo que se vio en el entrenamiento. Fuera de él, solo se **avisa**.
- **Rango posible** (`POSSIBLE_RANGE` en `config.py`): lo que puede ocurrir en el mundo real, **conocimiento del analista**.
  Saldos y montos no pueden ser negativos; porcentajes y razones van de 0 a 1; la edad, de 14 a 100 años (notebook 02).
  Fuera de él, la predicción **se niega** y explica por qué.

El esquema también lleva las etiquetas legibles de cada campo, qué variables son de Sí/No, la **regla de decisión** con sus
supuestos y tres **perfiles de ejemplo**: asociados reales del entrenamiento (sin datos personales) cuya probabilidad de
radicar es baja, intermedia y alta, para que quien abra la app empiece desde algo razonable.

In [4]:
complete = train.dropna(subset=RAW_INPUTS)
candidates = complete.sample(20_000, random_state=0)
p_candidates = model.predict_proba(candidates[INPUT_COLUMNS])[:, 1]

def profile(target_probability: float) -> dict:
    """El asociado de la muestra cuya probabilidad está más cerca de la pedida, como valores simples."""
    row = candidates.iloc[int(np.argmin(np.abs(p_candidates - target_probability)))]
    return {c: (int(row[c]) if c in BINARY_INPUTS else round(float(row[c]), 4)) for c in RAW_INPUTS}

examples = {"Perfil de bajo riesgo": profile(0.05), "Perfil de riesgo intermedio": profile(0.30),
            "Perfil de alto riesgo": profile(0.75)}

schema = build_schema(train, numeric=RAW_INPUTS, categorical=[], class_names=["no radica", "radica"],
                      valid_range={c: POSSIBLE_RANGE[c] for c in RAW_INPUTS}, labels=INPUT_LABELS)
schema.update({
    "binary_columns": BINARY_INPUTS,
    "examples": examples,
    "decision_rule": {"type": "valor_esperado", "value_column": "saldo_aportes", "contact_cost": CONTACT_COST,
                      "retention_rate": RETENTION_RATE, "label_yes": "contactar", "label_no": "no contactar",
                      "label_unknown": "revisar: falta el saldo de aportes"},
    "app_title": "Priorización de contacto para la retención de asociados",
    "app_subtitle": ("Estima la probabilidad de que un asociado radique la solicitud de retiro total y sugiere si conviene "
                     "contactarlo antes. **Apoya la decisión del equipo de fidelización; no decide por sí sola.**"),
    "decision_explanation": (
        f"El modelo estima la probabilidad de que el asociado radique la solicitud de retiro total. Se sugiere **contactar** "
        f"si *probabilidad × {RETENTION_RATE:.0%} (efectividad de la retención) × saldo de aportes* supera el costo de un "
        f"contacto (**{CONTACT_COST:g} millones**, es decir ${CONTACT_COST * 1e6:,.0f}). Así, dos asociados con la misma ".replace(",", ".")
        + "probabilidad no valen lo mismo si uno tiene diez veces más aportes. Los supuestos de costo y efectividad deben "
          "confirmarse con un piloto con grupo de control."),
    "responsible_use": (
        "- La herramienta **prioriza a quién contactar**; la decisión y la conversación son del asesor.\n"
        "- **No** debe usarse para negar productos o servicios a ningún asociado.\n"
        "- Describe asociaciones, no causas: no dice *por qué* se iría un asociado.\n"
        "- En asociados con menos de 7 meses de antigüedad encuentra solo a la mitad de quienes radican: para ellos, usar "
        "también una estrategia de bienvenida.\n"
        "- Límites completos en `models/ficha_modelo.md`."),
    "model": {"registro": MODEL_NAME, "version": registry["version"], "alias": registry["alias"],
              "decisiones_congeladas": decisions["fecha_y_hora"]},
})
save_schema(schema, MODELS_DIR / "input_schema.json")
schema = load_schema(MODELS_DIR / "input_schema.json")         # se trabaja con lo que quedó guardado
print(f"Esquema guardado: {len(schema['numeric'])} entradas ({len(schema['binary_columns'])} de Sí/No), "
      f"clases {schema['class_names']}, {len(schema['examples'])} perfiles de ejemplo")
display(pd.DataFrame(schema["examples"]).T[["saldo_aportes", "antig_meses", "porc_meses_publiturno",
                                           "porc_meses_capitaliza_aportes", "prom_saldo_cartera", "edad"]])

Esquema guardado: 19 entradas (4 de Sí/No), clases ['no radica', 'radica'], 3 perfiles de ejemplo


,saldo_aportes,antig_meses,porc_meses_publiturno,porc_meses_capitaliza_aportes,prom_saldo_cartera,edad
Perfil de bajo riesgo,1.5320,35.0,0.7222,1.0000,6.4526,21.0
Perfil de riesgo intermedio,3.2510,37.0,0.2105,0.0526,5.5962,45.0
Perfil de alto riesgo,0.2889,36.0,0.1892,0.2703,0.0000,46.0


## 3. Probar la predicción

Los casos que pide la guía, con `predict_with_scores` (la misma función que usa la app):

- una fila **normal** → responde con la decisión, la probabilidad y el valor esperado;
- una fila **incompleta** (quien usa la app marcó "No sé") → también responde, porque el pipeline imputa;
- una fila con un dato **imposible** → **se niega y explica por qué**;
- un caso **límite**: el mismo asociado con un saldo apenas distinto cambia de decisión.

In [5]:
model = load_model(MODELS_DIR / "model.joblib")          # desde aquí, el archivo exportado (el que usará la app)
show_cols = ["decision", "proba_radica", "expected_value", "contact_cost", "needs_review"]
names = {"decision": "decisión", "proba_radica": "probabilidad de radicar", "expected_value": "valor esperado (millones)",
         "contact_cost": "costo del contacto (millones)", "needs_review": "revisar"}
def show(result: pd.DataFrame):
    """Muestra la salida de predict_with_scores con nombres en español (solo al mostrar)."""
    display(table(result.rename(columns=names), decimals=4, pct_cols=["probabilidad de radicar"]))

print("— Los tres perfiles de ejemplo —")
profiles = pd.DataFrame(schema["examples"]).T
answers = predict_with_scores(profiles, model, schema)[show_cols]
answers.insert(1, "saldo de aportes (millones)", profiles["saldo_aportes"].astype(float))
show(answers)

normal = pd.DataFrame([schema["examples"]["Perfil de riesgo intermedio"]])
incomplete = normal.assign(ingresos=np.nan, prom_saldo_cartera=np.nan)
print("— Con dos datos desconocidos (ingresos y saldo de crédito: 'No sé') —")
show(predict_with_scores(incomplete, model, schema)[show_cols].rename(index={0: "perfil intermedio sin 2 datos"}))

print("— Con datos imposibles —")
for column, value in [("edad", 200), ("porc_meses_publiturno", 1.5), ("saldo_aportes", -2)]:
    try:
        predict_with_scores(normal.assign(**{column: value}), model, schema)
        raise AssertionError(f"debía negarse con {column} = {value}")
    except InputValidationError as error:
        print(f"  {column} = {value}: se niega, como debe → {error}")

print("— Caso límite: el perfil de alto riesgo con menos saldo de aportes —")
high = pd.DataFrame([schema["examples"]["Perfil de alto riesgo"]])
limit = pd.concat([predict_with_scores(high.assign(saldo_aportes=v), model, schema)[show_cols] for v in [0.25, 0.20]])
limit.index = ["saldo de aportes 0,25 millones", "saldo de aportes 0,20 millones"]
show(limit)

no_value = normal.assign(saldo_aportes=np.nan)
print("— Sin saldo de aportes —")
print("  decisión:", predict_with_scores(no_value, model, schema)["decision"].iloc[0])

— Los tres perfiles de ejemplo —


,decisión,saldo de aportes (millones),probabilidad de radicar,valor esperado (millones),costo del contacto (millones),revisar
Perfil de bajo riesgo,no contactar,"1,5320","5,0 %","0,0153","0,0300",False
Perfil de riesgo intermedio,contactar,"3,2510","29,7 %","0,1933","0,0300",False
Perfil de alto riesgo,contactar,"0,2889","75,0 %","0,0433","0,0300",False


— Con dos datos desconocidos (ingresos y saldo de crédito: 'No sé') —


,decisión,probabilidad de radicar,valor esperado (millones),costo del contacto (millones),revisar
perfil intermedio sin 2 datos,contactar,"33,2 %","0,2158","0,0300",False


— Con datos imposibles —
  edad = 200: se niega, como debe → edad: debe estar entre 14 y 100
  porc_meses_publiturno = 1.5: se niega, como debe → porc_meses_publiturno: debe estar entre 0 y 1
  saldo_aportes = -2: se niega, como debe → saldo_aportes: debe estar entre 0 y 10000
— Caso límite: el perfil de alto riesgo con menos saldo de aportes —


,decisión,probabilidad de radicar,valor esperado (millones),costo del contacto (millones),revisar
"saldo de aportes 0,25 millones",contactar,"75,0 %","0,0375","0,0300",False
"saldo de aportes 0,20 millones",no contactar,"74,4 %","0,0298","0,0300",False


— Sin saldo de aportes —
  decisión: revisar: falta el saldo de aportes


**Lectura.**

- **La regla decide con el saldo, no solo con la probabilidad.** El perfil de alto riesgo (75,0 %) tiene apenas 0,29
  millones de aportes: su contacto vale 0,0433 millones, poco más que el costo (0,03). El de riesgo intermedio (29,7 %)
  tiene 3,25 millones: su contacto vale 0,1933, **más de cuatro veces más**. El de bajo riesgo (5,0 %) vale 0,0153, la mitad
  del costo: no se contacta.
- **Con datos desconocidos responde igual.** Sin ingresos ni saldo de crédito, el perfil intermedio pasa de 29,7 % a 33,2 %
  y la decisión no cambia: el pipeline imputa con lo aprendido en el entrenamiento.
- **El caso límite muestra la frontera de la regla.** Con 0,25 millones de aportes el perfil de alto riesgo se contacta
  (valor esperado 0,0375); con 0,20 millones ya no (0,0298, por debajo del costo de 0,03), aunque su probabilidad casi no
  cambia (75,0 % y 74,4 %).
- **Con datos imposibles se niega y dice qué corregir** (edad de 200 años, una fracción de 1,5, un saldo negativo). Y **sin
  saldo de aportes no inventa una decisión**: la deja para revisión, porque sin saldo no hay valor esperado.
- Ninguno de los tres perfiles queda marcado para revisión: en los tres, la clase más probable supera el 55 %.

## 4. ¿El archivo exportado es el modelo que se evaluó?

Antes de dárselo a la app: `models/model.joblib`, llamado **por el mismo camino que usa la app** (`predict_with_scores` con
las 19 columnas que se digitan), debe dar **las mismas probabilidades** que se obtuvieron en la prueba del notebook 07 y la
**misma campaña**. Esto **no es una nueva lectura de la prueba para decidir algo**: es una comprobación de identidad contra
las probabilidades que ese notebook guardó cuando la abrió una sola vez.

In [6]:
test = joblib.load(PROCESSED_DIR / "test.joblib")
saved_test_proba, _ = joblib.load(PROCESSED_DIR / "07_prueba_probabilidades.joblib")
app_path = predict_with_scores(test[RAW_INPUTS], model, schema)
same_proba = bool(np.allclose(saved_test_proba, app_path["proba_radica"].to_numpy()))
contacts = int((app_path["decision"] == "contactar").sum())
identity = pd.DataFrame({
    "notebook 07": [len(test), results["campaña"]["contactados"]],
    "model.joblib por el camino de la app": [len(app_path), contacts],
}, index=["asociados de la prueba", "asociados a contactar (regla de valor esperado)"])
display(table(identity.astype(int)))
print(f"Probabilidades idénticas a las del notebook 07: {same_proba} "
      f"(diferencia máxima {np.abs(saved_test_proba - app_path['proba_radica'].to_numpy()).max():.2e})")
assert same_proba and contacts == results["campaña"]["contactados"]

,notebook 07,model.joblib por el camino de la app
asociados de la prueba,38.815,38.815
asociados a contactar (regla de valor esperado),15.930,15.930


Probabilidades idénticas a las del notebook 07: True (diferencia máxima 3.33e-16)


**Lectura.** El archivo que usará la app, consultado exactamente como lo hará la app (validando, calculando los indicadores y
aplicando la regla), reproduce las probabilidades y la campaña del notebook 07. Todo lo que dice la validación vale para lo
que se entrega.

## 5. La ficha del modelo

Una ficha dice **qué es**, **cuánto rinde (con su intervalo)**, **dónde falla** y **para qué no usarlo**. Se genera con las
cifras de `07_resultados.json`, `07_errores_por_segmento.csv`, `07_importancia_shap.csv` y `08_registro.json`, no a mano,
para que no pueda contradecir a los notebooks 07 y 08.

In [7]:
iv = {(r["qué"], r["métrica"]): r for r in results["intervalos"]}
def interval(what, metric, d=3):
    r = iv[(what, metric)]
    return (f"{fmt_num(r['valor'], d)} | {fmt_num(r['IC 95 % inferior'], d)} a {fmt_num(r['IC 95 % superior'], d)}")
m, camp = results["metricas_modelo"], results["campaña"]
segments = pd.read_csv(REPORTS_DIR / "07_errores_por_segmento.csv", index_col=[0, 1])
new_members = segments.loc[("antigüedad", "0–6 meses")]
shap_top = pd.read_csv(REPORTS_DIR / "07_importancia_shap.csv", index_col=0).head(6)
params = decisions["parametros"]

card = f"""# Ficha del modelo — Priorización de contacto para la retención de asociados

> Generada por `notebooks/09_inferencia.ipynb` el {datetime.now():%Y-%m-%d} con las cifras de los notebooks 07 y 08.

## Qué es

Estima la probabilidad de que un asociado de Confiar **radique la solicitud de retiro total de aportes**, para que el equipo
de fidelización **priorice a quién contactar antes**. Apoya la decisión; **no decide por sí solo**.

- **Modelo:** Random Forest sin calibrar ({params['n_estimators']} árboles, profundidad máxima {params['max_depth']}, hojas de al menos
  {params['min_samples_leaf']} asociados, {fmt_pct(params['max_features'], 0)} de las variables por corte).
- **Registro:** `{registry['modelo_registrado']}` versión {registry['version']}, alias `champion` (MLflow {registry['donde']}).
  Exportado a `models/model.joblib`.
- **Regla de decisión:** contactar si *probabilidad × {fmt_pct(RETENTION_RATE, 0)} × saldo de aportes > {fmt_num(CONTACT_COST, 2)}
  millones* (valor esperado del contacto mayor que su costo). Supuestos de costo y efectividad **por validar**.

## Datos de entrenamiento

- **Fuente:** datos internos de Confiar anonimizados por el equipo (`bd_confiar_2.xlsx`); asociados vinculados entre 2017
  y 2025, un registro por asociado. Montos en millones de pesos.
- **Tamaño:** {fmt_num(len(train))} asociados de entrenamiento; prueba de {fmt_num(results['prueba']['asociados'])} asociados.
- **Partición:** por asociado, estratificada; la prueba se abrió **una sola vez** (decisiones congeladas el
  {decisions['fecha_y_hora']}).
- **Datos personales:** el `Nit` se eliminó al cargar; las fechas del registro se eliminaron por fuga.

## Rendimiento (prueba, asociados nunca vistos)

| Métrica | Valor | Intervalo del 95 % (remuestrea asociados) |
|---|---|---|
| Beneficio neto por cada 1.000 asociados (millones, principal) | {interval('modelo final', 'beneficio neto (por 1.000)', 2)} |
| Beneficio frente a contactar a todos (modelo − contactar a todos) | {interval('modelo − contactar a todos', 'beneficio neto (por 1.000)', 2)} |
| PR-AUC (identificación de quien radica) | {interval('modelo final', 'PR-AUC')} |
| PR-AUC frente al baseline (modelo − baseline) | {interval('modelo − baseline', 'PR-AUC')} |
| AUC | {interval('modelo final', 'AUC')} |
| Brier (calidad de las probabilidades) | {interval('modelo final', 'Brier')} |

Con la regla de valor esperado contacta al {fmt_pct(camp['contactados'] / results['prueba']['asociados'], 0)} de los asociados,
encuentra al {fmt_pct(camp['VP'] / results['prueba']['radicaron'], 1)} de quienes radican y cubre el {fmt_pct(camp['saldo_cubierto'], 1)} del saldo
de aportes en riesgo.

## Variables que más pesan (SHAP)

{chr(10).join(f"- `{name}`: {fmt_pct(row['% del total'], 1)} de la influencia ({'valores altos suben' if row['dirección'] > 0 else 'valores altos bajan'} el riesgo)" for name, row in shap_top.iterrows())}

Explican **al modelo, no al mundo**: no son causas del retiro.

## Límites y usos no recomendados

- **Asociados nuevos (0 a 6 meses):** encuentra solo al {fmt_pct(new_members['recall'], 1)} de quienes radican (poca historia y poco
  saldo). Usar también una estrategia de bienvenida.
- **Sospechas de fuga:** visitas a sucursal, cartera mínima en cero, ingreso y tendencias de saldo suman cerca del 40 % de la
  influencia. El equipo confirmó que no incluyen el mes de radicación; falta aclarar la ventana de cálculo. Sin ellas el modelo
  seguía superando a contactar a todos (notebook 05).
- **Supuestos de negocio:** las cifras en pesos dependen del costo del contacto ($30.000) y de la efectividad de la retención
  (20 %). Validarlos con un piloto con grupo de control.
- **Capacidad:** la regla contacta a cerca del 41 % de los asociados; confirmar que mercadeo puede hacerlo.
- **Atributos sensibles** (edad, sexo, zona, educación, ocupación): pesan poco y el error por segmento es parecido entre
  grupos, pero deben revisarse con el área jurídica antes de producción.
- **Qué no hacer:** negar productos o servicios; leer las variables como causas; decidir sobre grupos sociodemográficos.
- **No se evaluó:** estabilidad en el tiempo (deriva), comportamiento con nuevas cohortes o productos, efecto real de la
  campaña. Recalibrar y revisar cada seis meses.

## Reproducibilidad

Código en `src/proyecto/`, pruebas en `tests/` (`pytest -q`), notebooks 01 a 09 en `notebooks/`. App: `streamlit run
app/streamlit_app.py` (carga `models/model.joblib` y `models/input_schema.json`; no consulta Databricks).
"""
(MODELS_DIR / "ficha_modelo.md").write_text(card, encoding="utf-8")
print(card)

# Ficha del modelo — Priorización de contacto para la retención de asociados

> Generada por `notebooks/09_inferencia.ipynb` el 2026-11-06 con las cifras de los notebooks 07 y 08.

## Qué es

Estima la probabilidad de que un asociado de Confiar **radique la solicitud de retiro total de aportes**, para que el equipo
de fidelización **priorice a quién contactar antes**. Apoya la decisión; **no decide por sí solo**.

- **Modelo:** Random Forest sin calibrar (200 árboles, profundidad máxima 14, hojas de al menos
  15 asociados, 27 % de las variables por corte).
- **Registro:** `retiro_asociados_rf` versión 1, alias `champion` (MLflow local).
  Exportado a `models/model.joblib`.
- **Regla de decisión:** contactar si *probabilidad × 20 % × saldo de aportes > 0,03
  millones* (valor esperado del contacto mayor que su costo). Supuestos de costo y efectividad **por validar**.

## Datos de entrenamiento

- **Fuente:** datos internos de Confiar anonimizados por el equipo (`bd_confiar_2.xlsx`); as

## 6. ¿Está todo modularizado y listo para la app?

La regla del curso desde la clase 2: **la lógica reutilizable vive en `src/`; los notebooks solo orquestan.** Se revisa
módulo por módulo, se corren **todas las pruebas** y se **abre la app con el modelo real** usando el probador oficial de
Streamlit (sin navegador).

In [8]:
src = ROOT / "src" / "proyecto"
modules = pd.DataFrame([
    ("config.py", "rutas, columnas, reglas de limpieza, rangos posibles y supuestos del negocio", "—"),
    ("data/cleaning.py", "carga sin datos personales y limpieza (notebooks 02-03)", "test_cleaning.py"),
    ("data/split.py", "partición y pliegues por asociado (03-06)", "test_split.py, test_models.py"),
    ("features/engineering.py", "variables nuevas y preprocesamiento en el Pipeline (04)", "test_features.py"),
    ("models/business.py", "métricas de negocio y regla de valor esperado (01, 05)", "test_business.py"),
    ("models/train.py", "candidatos, métricas y validación cruzada (04-06)", "test_models.py"),
    ("models/tuning.py", "optimización con Optuna y MLflow (06)", "test_tuning.py"),
    ("models/validation.py", "intervalos, calibración, segmentos y SHAP (07)", "test_validation.py"),
    ("tracking/mlflow_utils.py", "conexión con MLflow / Databricks (06, 08)", "test_tracking.py"),
    ("inference/predict.py", "validar, predecir y aplicar la regla (09 y app)", "test_inference.py, test_predict_retiro.py"),
    ("inference/schema.py", "esquema de entrada (09 y app)", "test_inference.py"),
    ("viz.py", "estilo de gráficos y formato en español (todos)", "test_viz.py"),
], columns=["módulo de src/proyecto", "qué hace (notebooks)", "pruebas"]).set_index("módulo de src/proyecto")
modules["existe"] = [(src / m).exists() for m in modules.index]
display(modules)
assert modules["existe"].all()

run = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no"], cwd=ROOT,
                     capture_output=True, text=True, encoding="utf-8")
summary = [line for line in run.stdout.splitlines() if "passed" in line or "failed" in line or "error" in line.lower()]
print("pytest:", summary[-1] if summary else run.stdout[-500:])
assert run.returncode == 0, run.stdout[-2000:]

,qué hace (notebooks),pruebas,existe
módulo de src/proyecto,,,
config.py,"rutas, columnas, reglas de limpieza, rangos po...",—,True
data/cleaning.py,carga sin datos personales y limpieza (noteboo...,test_cleaning.py,True
data/split.py,partición y pliegues por asociado (03-06),"test_split.py, test_models.py",True
features/engineering.py,variables nuevas y preprocesamiento en el Pipe...,test_features.py,True
models/business.py,métricas de negocio y regla de valor esperado ...,test_business.py,True
models/train.py,"candidatos, métricas y validación cruzada (04-06)",test_models.py,True
models/tuning.py,optimización con Optuna y MLflow (06),test_tuning.py,True
models/validation.py,"intervalos, calibración, segmentos y SHAP (07)",test_validation.py,True
tracking/mlflow_utils.py,"conexión con MLflow / Databricks (06, 08)",test_tracking.py,True


pytest: 75 passed, 1 warning in 41.20s


In [9]:
from streamlit.testing.v1 import AppTest

# sin el aviso de "modo sin navegador" del probador (Streamlit le fija nivel propio a cada registro)
for name in ["streamlit", "streamlit.runtime.scriptrunner_utils.script_run_context"]:
    logging.getLogger(name).setLevel(logging.ERROR)

app = AppTest.from_file(str(ROOT / "app" / "streamlit_app.py"), default_timeout=180).run()
assert not app.exception, app.exception
print("La app abre sin errores:", app.title[0].value)
rows = []
for name in schema["examples"]:
    app.selectbox[0].set_value(name).run()
    app.button[0].click().run()
    assert not app.exception, app.exception
    metrics = {m.label: m.value for m in app.metric}
    rows.append({"perfil": name, "decisión que muestra la app": metrics["Decisión sugerida"],
                 "probabilidad que muestra la app": metrics["Probabilidad de que radica"]})
display(pd.DataFrame(rows).set_index("perfil"))

La app abre sin errores: Priorización de contacto para la retención de asociados


,decisión que muestra la app,probabilidad que muestra la app
perfil,,
Perfil de bajo riesgo,No contactar,"5,0%"
Perfil de riesgo intermedio,Contactar,"29,7%"
Perfil de alto riesgo,Contactar,"75,0%"


In [10]:
checks = {
    "requirements.txt está en la raíz": (ROOT / "requirements.txt").exists(),
    "streamlit está en requirements.txt": "streamlit" in (ROOT / "requirements.txt").read_text(encoding="utf-8"),
    "models/model.joblib no está en .gitignore": subprocess.run(["git", "check-ignore", "-q", "models/model.joblib"], cwd=ROOT).returncode == 1,
    "models/input_schema.json no está en .gitignore": subprocess.run(["git", "check-ignore", "-q", "models/input_schema.json"], cwd=ROOT).returncode == 1,
    "models/model.joblib pesa menos de 100 MB": (MODELS_DIR / "model.joblib").stat().st_size < 100e6,
    "el pipeline intermedio (final_pipeline.joblib) sí está en .gitignore": subprocess.run(["git", "check-ignore", "-q", "models/final_pipeline.joblib"], cwd=ROOT).returncode == 0,
    "los datos (data/raw, data/processed) están en .gitignore": all(subprocess.run(["git", "check-ignore", "-q", f], cwd=ROOT).returncode == 0
                                                                for f in ["data/raw/bd_confiar_2.xlsx", "data/processed/train.joblib"]),
    "el registro local de MLflow (mlflow_local.db) está en .gitignore": subprocess.run(["git", "check-ignore", "-q", "mlflow_local.db"], cwd=ROOT).returncode == 0,
    "la app no usa Databricks ni credenciales": "databricks" not in (ROOT / "app" / "streamlit_app.py").read_text(encoding="utf-8").lower().replace("no consulta databricks", ""),
}
ready = pd.Series(checks, name="se cumple").to_frame()
display(ready)
assert ready["se cumple"].all()

,se cumple
requirements.txt está en la raíz,True
streamlit está en requirements.txt,True
models/model.joblib no está en .gitignore,True
models/input_schema.json no está en .gitignore,True
models/model.joblib pesa menos de 100 MB,True
el pipeline intermedio (final_pipeline.joblib) sí está en .gitignore,True
"los datos (data/raw, data/processed) están en .gitignore",True
el registro local de MLflow (mlflow_local.db) está en .gitignore,True
la app no usa Databricks ni credenciales,True


**Lectura.** Los 12 módulos de `src/proyecto` existen y todos, salvo `config.py` (solo constantes), tienen pruebas; las
**75 pruebas pasan**, incluidas las que usan el modelo exportado y la que abre la app. La app, abierta con el probador de
Streamlit, muestra para cada perfil **la misma decisión y la misma probabilidad** que `predict_with_scores` en la sección 3
(5,0 %, 29,7 % y 75,0 %): no tiene lógica propia, solo dibuja el formulario a partir del esquema y llama a `predict`. Los
dos archivos que necesita viajan con el repositorio (`model.joblib` pesa 17,6 MB); los datos, el pipeline intermedio de
54,6 MB y el registro local de MLflow no salen del computador.

## 7. Conclusiones

1. **El modelo ya funciona sin MLflow y es el mismo que se evaluó.** `models/model.joblib` (17,6 MB, exportado desde el
   alias `champion`) y `models/input_schema.json` reproducen las probabilidades de la prueba del notebook 07 (diferencia
   máxima de 3,3 × 10⁻¹⁶) y la misma campaña: 15.930 contactos entre 38.815 asociados.
2. **`predict.py` sí era necesario: es la pieza central de la inferencia.** Lo usan este notebook, las pruebas y la app.
   Se adaptó en dos puntos: calcula los 3 indicadores de tenencia (la app pide 19 datos, no 22) y aplica la regla de valor
   esperado del caso de negocio en vez de la clase más probable.
3. **Valida antes de responder.** Se niega con datos imposibles y explica por qué, responde con datos desconocidos y deja
   para revisión a quien no tiene saldo de aportes.
4. **La decisión depende del saldo.** Un asociado con 29,7 % de probabilidad y 3,25 millones de aportes vale más de cuatro
   veces lo que vale uno con 75,0 % y 0,29 millones. Por eso la app muestra el valor esperado, no solo la probabilidad.
5. **La ficha del modelo (`models/ficha_modelo.md`) sale de los resultados, no se escribe a mano.** Beneficio neto de 33,60
   millones por cada 1.000 asociados (IC 95 %: 31,81 a 35,40), 14,81 millones más que contactar a todos (14,61 a 15,00) y
   PR-AUC de 0,700 (0,690 a 0,711). Deja escritos sus límites: con los asociados nuevos encuentra solo al 49,2 % de quienes
   radican, cerca del 40 % de la influencia está en variables sospechosas de fuga y las cifras en pesos dependen de dos
   supuestos por validar.
6. **El código está modularizado y la app está lista para desplegarse.** La lógica vive en 12 módulos de `src/` con 75
   pruebas que pasan; la app abre con el modelo real y no necesita credenciales.

**Siguiente paso: el despliegue.**

1. En local, desde la raíz del repositorio: `streamlit run app/streamlit_app.py`.
2. En Streamlit Community Cloud: conectar el repositorio de GitHub, rama `main`, archivo principal
   `app/streamlit_app.py`, Python 3.12 (en la configuración avanzada). La nube instala `requirements.txt` desde la raíz.

**Pendiente para el equipo antes de usarla con asociados reales:** validar el costo del contacto y la efectividad de la
retención con un piloto con grupo de control, confirmar que mercadeo puede contactar a cerca del 41 % de los asociados y
aclarar la ventana de cálculo de las variables sospechosas de fuga.